# MiniSense — End-to-End Workflow

Runs the whole pipeline live, in-process, against the local Ollama server: load the survey dataset, call the `DataAgent` tool functions directly, retrieve FAQ chunks with `RAGAgent`, then run the full `Orchestrator -> sub-agents -> SummaryAgent` flow for a real business question.

Prerequisites (see README section 3, Option A): `ollama serve` running, `llama3.1:8b` and `nomic-embed-text` pulled, `data/survey_responses.json` generated, and the FAQ index built via `scripts/ingest_faq.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from minisense.config import get_settings
from minisense.data_loader import load_responses
from minisense.llm.ollama_client import is_available

settings = get_settings()
print("Survey path:", settings.survey_json_path)
print("LLM model:", settings.llm_model, " Embed model:", settings.embed_model)
print("Ollama reachable:", is_available())

Survey path: D:\minisense\data\survey_responses.json
LLM model: llama3.1:8b  Embed model: nomic-embed-text
Ollama reachable: True


## 1. Load and inspect the survey dataset

In [2]:
responses = load_responses()
print(f"{len(responses):,} validated responses loaded")
responses[0]

ts=2026-09-05T09:52:36+0530 level=INFO logger=minisense.data_loader msg="Loaded 100000 valid survey response(s) from D:\minisense\data\survey_responses.json (0 skipped)"


100,000 validated responses loaded


{'response_id': 'r000001',
 'date': '2026-05-11',
 'business_id': 'b01',
 'business_name': 'GreenLeaf Bistro - Downtown',
 'survey_id': 's01',
 'survey_name': 'Dine-In Experience',
 'rating': 5,
 'response_channel': 'mobile',
 'free_text': 'the grilled salmon salad arrived hot and tasted great. that said, line was out the door and moved painfully slowly...'}

## 2. Tool calling from within an agent

`DataAgent` is LLM-free — it calls plain, tested Python functions in `tools/metrics.py` directly. This is the assignment's required "tool calling from within an agent" example.

In [3]:
from minisense.tools import metrics

print("compute_csat:", metrics.compute_csat(responses), "%")
print("compute_average_rating:", metrics.compute_average_rating(responses))
print("compute_top_themes:", metrics.compute_top_themes(responses, top_n=5))

compute_csat: 62.3 %
compute_average_rating: 3.649


compute_top_themes: [('staff', 8036), ('food_quality', 7550), ('wait_time', 6385), ('cleanliness', 4388), ('value', 2975)]


## 3. RAGAgent — retrieve grounded FAQ context

In [4]:
from minisense.agents import rag_agent
from minisense.schemas import AgentName, TaskSpec

rag_task = TaskSpec(agent=AgentName.RAG, objective="CSAT policy context", query_text="What is the CSAT target?", top_k=3)
rag_result = rag_agent.run(rag_task)
for c in rag_result.chunks:
    print(f"[{c.score:.3f}] {c.chunk_id}: {c.text[:160]}...")

[0.737] chunk_007: **Q: What is your CSAT target?** A: We aim for a CSAT of 4.5+ (share of survey responses rated 4 or 5 out of 5). Scores below 4.0 in any rolling 30-day window t...
[0.437] chunk_012: **Q: Do you operate multiple locations?** A: Yes. GreenLeaf Bistro operates as a small multi-location business, and each location is tracked separately in our s...
[0.432] chunk_004: **Q: What is your average wait time?** A: We target under 10 minutes for counter orders during off-peak hours. Peak hours (12–1 PM and 6–8 PM) may see 15–20 min...


## 4. Full pipeline — Orchestrator to SummaryAgent

One call drives the whole multi-agent flow: plan -> route to sub-agents -> synthesize a narrative answer.

In [5]:
from minisense.agents.orchestrator import answer_question

question = "What is our overall CSAT and how does it compare to our stated CSAT target?"
run = answer_question(question, responses)

print("PLAN REASONING:", run.plan.reasoning)
for t in run.plan.tasks:
    print(" ->", t.agent.value, "-", t.objective)
print()
print("FINAL ANSWER:\n")
print(run.summary.narrative)

ts=2026-09-05T09:53:21+0530 level=INFO logger=minisense.agents.orchestrator msg="LLM plan produced 3 task(s) for question: 'What is our overall CSAT and how does it compare to our stated CSAT target?'"


ts=2026-09-05T09:54:19+0530 level=INFO logger=minisense.agents.orchestrator msg="Answered question with 4 agent step(s): 'What is our overall CSAT and how does it compare to our stated CSAT target?'"


PLAN REASONING: Break down the question into two tasks: one to compute the current CSAT and another to compare it to the target.
 -> DataAgent - Compute the current CSAT
 -> RAGAgent - Retrieve CSAT target and policy context
 -> ComparisonAgent - Compare current CSAT to target

FINAL ANSWER:

Our overall CSAT for the month of May 2026 was 62.13%, which falls short of our target of 4.5+ (a share of survey responses rated 4 or 5 out of 5). While our CSAT has increased by 0.27% compared to the previous period, this is not significant enough to meet our target. We did see a notable increase in customer complaints about wait times, with mentions up 50% compared to the previous period. Additionally, we experienced a significant increase in response count, with 45857 more responses than the previous period, which may be due to the fact that we operate as a small multi-location business and each location is tracked separately. We will need to review our operations to address the issues driving

## 5. Comparative question — ComparisonAgent in the loop

In [6]:
run2 = answer_question(
    "What are the top 3 complaints this month and how do they compare to last month?",
    responses,
)
if run2.comparison_result:
    for d in run2.comparison_result.deltas:
        flag = "significant" if d.is_significant else "not significant"
        print(f"{d.metric}: {d.period_a_value} -> {d.period_b_value} ({flag})")
print()
print(run2.summary.narrative)

ts=2026-09-05T09:54:30+0530 level=INFO logger=minisense.agents.orchestrator msg="LLM plan produced 2 task(s) for question: 'What are the top 3 complaints this month and how do they compare to last month?'"


ts=2026-09-05T09:54:57+0530 level=INFO logger=minisense.agents.orchestrator msg="Answered question with 3 agent step(s): 'What are the top 3 complaints this month and how do they compare to last month?'"


average_rating: 3.655 -> 3.645 (not significant)
csat_pct: 62.5 -> 62.13 (not significant)
response_count: 45857.0 -> 54143.0 (not significant)

This month, the top three complaints were about wait time, staff, and food quality, with 4,248, 4,234, and 4,221 mentions respectively. Compared to last month, food quality complaints increased by 27%, while wait time complaints rose by a significant 99%. On the other hand, average ratings and customer satisfaction percentages decreased by 0.27% and 0.59%, respectively. The number of responses also increased by 18% to 54,143. It's worth noting that according to our FAQ, we aim to maintain an average rating of at least 4, but this month's rating fell short of that target.


## 6. Full structured trace (what the API's `--trace` / `/ask` response exposes)

In [7]:
import json

print(json.dumps([{"agent": step.agent, "result": step.result} for step in run.trace], indent=2, default=str)[:3000])

[
  {
    "agent": "DataAgent",
    "result": {
      "period": {
        "start": "2026-05-01",
        "end": "2026-05-31"
      },
      "business_id": null,
      "response_count": 54143,
      "average_rating": 3.645,
      "csat_pct": 62.13,
      "top_themes": [
        {
          "theme": "wait_time",
          "count": 4248
        },
        {
          "theme": "staff",
          "count": 4234
        },
        {
          "theme": "food_quality",
          "count": 4221
        }
      ],
      "channel_breakdown": {
        "mobile": 21707,
        "email": 6593,
        "web": 11888,
        "kiosk": 9599,
        "in_store_tablet": 4356
      }
    }
  },
  {
    "agent": "RAGAgent",
    "result": {
      "query": "CSAT target",
      "chunks": [
        {
          "chunk_id": "chunk_007",
          "text": "**Q: What is your CSAT target?** A: We aim for a CSAT of 4.5+ (share of survey responses rated 4 or 5 out of 5). Scores below 4.0 in any rolling 30-day window tri